<a href="https://colab.research.google.com/github/jc020230/practiceai/blob/main/4%EC%9E%A5_1_querying_and_merging.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Performing Database-style Operations on Dataframes

## About the data
In this notebook, we will using daily weather data that was taken from the [National Centers for Environmental Information (NCEI) API](https://www.ncdc.noaa.gov/cdo-web/webservices/v2). The [`0-weather_data_collection.ipynb`](./0-weather_data_collection.ipynb) notebook contains the process that was followed to collect the data. Consult the dataset's [documentation](https://www1.ncdc.noaa.gov/pub/data/cdo/documentation/GHCND_documentation.pdf) for information on the fields.

*Note: The NCEI is part of the National Oceanic and Atmospheric Administration (NOAA) and, as you can see from the URL for the API, this resource was created when the NCEI was called the NCDC. Should the URL for this resource change in the future, you can search for "NCEI weather API" to find the updated one.*


## Setup

**요약**

- `query('조건')` → 필터링을 문자열로 쓰는 방법. `df[(df.a == 1) & (df.b > 0)]` 대신 `df.query('a == 1 and b > 0')`. 결과는 똑같고 읽기만 편함
- `merge()` → 표 두 개를 '공통 열'을 기준으로 옆으로 합치기 (중요할듯)
  - inner(기본): 양쪽에 다 있는 것만 남김
  - left: 왼쪽 표는 전부 남기고, 오른쪽에서 짝 없으면 NaN
  - right: 오른쪽 표는 전부 남김
  - outer: 양쪽 다 전부 남김
- `join()` → merge인데 인덱스 기준으로 합칠 때 쓰는 간편 버전
- 뒤쪽 집합 연산(intersection, difference, union)은 merge하기 전에 뭐가 겹치고 뭐가 빠지는지 미리 확인하는 도구

merge할 때 항상 행 개수(shape)를 전후로 비교하는 게 중요함. 행이 확 줄거나 늘면 뭔가 잘못된 거임

코드 보니까 데이터베이스 설계같은거 할 때 나오는 코드들인데, inner join left join right join이런거 구분하는거 나올거 같아

**쉽게 말하면, query()는 복잡한 조건들을 한 줄로 깔끔하게 표현할 수 있게 하는 도구임**

In [ ]:
import pandas as pd

# 2018년 뉴욕 날씨 데이터. 관측소(station)가 여러 개라서 같은 날짜가 여러 행
# 열: date(날짜), datatype(측정 종류: PRCP 강수량, SNOW 적설량 등), station(관측소 ID), attributes(품질 표시), value(값)
weather = pd.read_csv('data/nyc_weather_2018.csv')
weather.head()


,date,datatype,station,attributes,value
0,2018-01-01T00:00:00,PRCP,GHCND:US1CTFR0039,",,N,",0.0
1,2018-01-01T00:00:00,PRCP,GHCND:US1NJBG0015,",,N,",0.0
2,2018-01-01T00:00:00,SNOW,GHCND:US1NJBG0015,",,N,",0.0
3,2018-01-01T00:00:00,PRCP,GHCND:US1NJBG0017,",,N,",0.0
4,2018-01-01T00:00:00,SNOW,GHCND:US1NJBG0017,",,N,",0.0


## Querying DataFrames
The `query()` method is an easier way of filtering based on some criteria. For example, we can use it to find all entries where snow was recorded from a station with `US1NY` in its station ID:

In [ ]:
# .query('조건') → 조건을 문자열로 써서 필터링. 2장에서 배운 불리언 마스크와 결과는 같음
# 'datatype == "SNOW" and value > 0 and station.str.contains("US1NY")'
#   → 눈(SNOW) 데이터이고 AND 값이 0보다 크고 AND 관측소 ID에 US1NY(뉴욕)가 들어간 행
# query 안에서는 and/or 를 그대로 쓸 수 있음 (마스크 방식에서는 &/| 써야 했던 것과 차이)
# 문자열 안에 문자열 넣을 땐 바깥은 ' ', 안쪽은 " " 로 따옴표를 다르게
snow_data = weather.query('datatype == "SNOW" and value > 0 and station.str.contains("US1NY")')
snow_data.head()


,date,datatype,station,attributes,value
114,2018-01-01T00:00:00,SNOW,GHCND:US1NYWC0019,",,N,",25.0
789,2018-01-04T00:00:00,SNOW,GHCND:US1NYNS0007,",,N,",41.0
794,2018-01-04T00:00:00,SNOW,GHCND:US1NYNS0018,",,N,",10.0
798,2018-01-04T00:00:00,SNOW,GHCND:US1NYNS0024,",,N,",89.0
800,2018-01-04T00:00:00,SNOW,GHCND:US1NYNS0030,",,N,",102.0


This is equivalent to querying the `weather.db` SQLite database for

```sql
SELECT *
FROM weather
WHERE datatype == "SNOW" AND value > 0 AND station LIKE "%US1NY%"
```

In [ ]:
import sqlite3

# 같은 걸 SQL로 하면 이렇게 됨. query()가 SQL의 WHERE와 거의 같다는 걸 보여주는 거임 -> SQL은 파이썬 같은 다른 도구임 신경쓰지마셈
# SELECT * FROM weather WHERE ... → weather 테이블에서 조건에 맞는 것 전부 가져와
# LIKE "%US1NY%" → SQL에서 '포함하나?' 검사. % 는 아무 글자
with sqlite3.connect('data/weather.db') as connection:
    snow_data_from_db = pd.read_sql(
        'SELECT * FROM weather WHERE datatype == "SNOW" AND value > 0 and station LIKE "%US1NY%"',
        connection
    )

# .reset_index().drop(columns='index') → query 결과는 원래 인덱스(114, 789...)가 남아있어서 0,1,2로 새로 매기고 옛 인덱스 열은 버림
# DB에서 읽은 건 인덱스가 0,1,2라서 이렇게 맞춰야 equals 비교 가능 → True (내용 똑같음)
# pd.read_sql()로 DB에서 읽어온 결과는 인덱스가 0,1,2,…로 새로 매겨져 있음.
# 반면 query()로 필터링한 DataFrame은 원래 인덱스(ex: 114, 789…)가 유지됨.
snow_data.reset_index().drop(columns='index').equals(snow_data_from_db)

# 결론: query()를 통해 내가 필요한 조건에 맞는 것만 가져올 수 있음!!!!!!!!!!!

True

Note this is also equivalent to creating Boolean masks:

In [ ]:
# 2장에서 배운 불리언 마스크 방식으로 똑같이. 조건마다 괄호 + & 로 연결
# .equals(snow_data) → query()로 한 것과 같은가? → True
# 결국 query / SQL / 마스크 셋 다 같은 일을 하는 거임. query가 제일 짧고 읽기 편함
weather[
    (weather.datatype == 'SNOW')
    & (weather.value > 0)
    & weather.station.str.contains('US1NY')
].equals(snow_data)


True

## Merging DataFrames
We have data for many different stations each day; however, we don't know what the stations are&mdash;just their IDs. We can join the data in the `weather_stations.csv` file which contains information from the `stations` endpoint of the NCEI API. Consult the [`0-weather_data_collection.ipynb`](./0-weather_data_collection.ipynb) notebook to see how this was collected. It looks like this:

In [ ]:
# 관측소 정보 파일. id(관측소 ID), name(이름), latitude(위도), longitude(경도), elevation(고도)
# weather에는 관측소 ID만 있고 이름이 없으니까, 이 표를 붙여서 이름을 알아내려는 거임
station_info = pd.read_csv('data/weather_stations.csv')
station_info.head()


,id,name,latitude,longitude,elevation
0,GHCND:US1CTFR0022,"STAMFORD 2.6 SSW, CT US",41.064100,-73.577000,36.6
1,GHCND:US1CTFR0039,"STAMFORD 4.2 S, CT US",41.037788,-73.568176,6.4
2,GHCND:US1NJBG0001,"BERGENFIELD 0.3 SW, NJ US",40.921298,-74.001983,20.1
3,GHCND:US1NJBG0002,"SADDLE BROOK TWP 0.6 E, NJ US",40.902694,-74.083358,16.8
4,GHCND:US1NJBG0003,"TENAFLY 1.3 W, NJ US",40.914670,-73.977500,21.6


As a reminder, the weather data looks like this:

In [ ]:
# 날씨 데이터 다시 확인. station 열의 값(GHCND:US1CTFR0039)이 station_info의 id 열과 같은 형식
# -> 이 두 열을 열쇠(key)로 삼아서 두 표를 합칠 수 있음
weather.head()


,date,datatype,station,attributes,value
0,2018-01-01T00:00:00,PRCP,GHCND:US1CTFR0039,",,N,",0.0
1,2018-01-01T00:00:00,PRCP,GHCND:US1NJBG0015,",,N,",0.0
2,2018-01-01T00:00:00,SNOW,GHCND:US1NJBG0015,",,N,",0.0
3,2018-01-01T00:00:00,PRCP,GHCND:US1NJBG0017,",,N,",0.0
4,2018-01-01T00:00:00,SNOW,GHCND:US1NJBG0017,",,N,",0.0


We can join our data by matching up the `station_info.id` column with the `weather.station` column. Before doing that though, let's see how many unique values we have:

In [ ]:
# station_info의 id: 279개, unique(서로 다른 값)도 279개 -> 관측소 하나당 정확히 한 행 (중복 없음)
station_info.id.describe()


count                   279
unique                  279
top       GHCND:US1NJBG0029
freq                      1
Name: id, dtype: object

While `station_info` has one row per station, the `weather` dataframe has many entries per station. Notice it also has fewer uniques:

In [ ]:
# weather의 station: 78780행인데 unique는 110개 -> 관측소 110개가 각각 여러 행 (날짜별, 측정 종류별)
# station_info(279개)보다 관측소 수가 적음 -> 날씨 기록이 없는 관측소가 169개 있다는 뜻. 나중에 중요해짐
weather.station.describe()


count                 78780
unique                  110
top       GHCND:USW00094789
freq                   4270
Name: station, dtype: object

When working with joins, it is important to keep an eye on the row count. Some join types will lead to data loss. Remember that we can get this with `shape`:

In [ ]:
# 합치기 전 행 개수 확인. shape[0] = 행 수 -> (279, 78780)
# merge 종류에 따라 행이 사라지거나 늘어날 수 있어서, 전후로 행 수를 꼭 비교해야 함
station_info.shape[0], weather.shape[0]


(279, 78780)

Since we will be doing this often, it makes more sense to write a function:

In [ ]:
# 행 개수를 자주 볼 거니까 함수로 만듦
# *dfs -> 데이터프레임을 몇 개든 받을 수 있다는 뜻 (별표 = "여러 개 받을게"). dfs는 받은 것들의 튜플
# [df.shape[0] for df in dfs] -> 각각의 행 개수를 리스트로
def get_row_count(*dfs):
    return [df.shape[0] for df in dfs]
# station_info와 weather 두 개 넣으면 -> [279, 78780]
get_row_count(station_info, weather)


[279, 78780]

By default, `merge()` performs an inner join. We simply specify the columns to use for the join. The left dataframe is the one we call `merge()` on, and the right one is passed in as an argument:

In [ ]:
# .merge() -> 두 표를 옆으로 합치기. 중요합니다~~
# weather.merge(station_info, ...) -> weather가 왼쪽(left), station_info가 오른쪽(right)
# left_on='station', right_on='id' -> 왼쪽 표의 station 열과 오른쪽 표의 id 열이 같은 행끼리 짝을 맞춤
# 기본은 inner join -> 양쪽 모두에 있는 관측소만 남김

# 결과: weather의 각 행 옆에 그 관측소의 name, latitude, longitude 등이 붙음
inner_join = weather.merge(station_info, left_on='station', right_on='id')
inner_join.sample(5, random_state=0)


,date,datatype,station,attributes,value,id,name,latitude,longitude,elevation
10739,2018-08-07T00:00:00,SNOW,GHCND:US1NJMN0069,",,N,",0.0,GHCND:US1NJMN0069,"LONG BRANCH 1.7 SSW, NJ US",40.275368,-74.006027,9.4
45188,2018-12-21T00:00:00,TMAX,GHCND:USW00014732,",,W,2400",16.7,GHCND:USW00014732,"LAGUARDIA AIRPORT, NY US",40.779440,-73.880350,3.4
59823,2018-01-15T00:00:00,WDF5,GHCND:USW00094741,",,W,",40.0,GHCND:USW00094741,"TETERBORO AIRPORT, NJ US",40.850000,-74.061390,2.7
10852,2018-10-31T00:00:00,PRCP,GHCND:US1NJMN0069,"T,,N,",0.0,GHCND:US1NJMN0069,"LONG BRANCH 1.7 SSW, NJ US",40.275368,-74.006027,9.4
46755,2018-05-05T00:00:00,SNOW,GHCND:USW00014734,",,W,",0.0,GHCND:USW00014734,"NEWARK LIBERTY INTERNATIONAL AIRPORT, NJ US",40.682500,-74.169400,2.1


We can remove the duplication of information in the `station` and `id` columns by renaming one of them before the merge and then simply using `on`:

In [ ]:
# 위 결과를 보면 station 열과 id 열에 같은 값이 두 번 들어가 있어서 낭비임
# station_info.rename(dict(id='station'), axis=1) -> 오른쪽 표의 id 열 이름을 station으로 미리 바꿈
#   (dict(id='station') 은 {'id': 'station'} 과 같음. 딕셔너리 만드는 다른 문법)
# on='station' -> 양쪽 다 열 이름이 station이니까 left_on/right_on 대신 on 하나로 씀
# -> 결과에 station 열이 하나만 남음
weather.merge(station_info.rename(dict(id='station'), axis=1), on='station').sample(5, random_state=0)


,date,datatype,station,attributes,value,name,latitude,longitude,elevation
10739,2018-08-07T00:00:00,SNOW,GHCND:US1NJMN0069,",,N,",0.0,"LONG BRANCH 1.7 SSW, NJ US",40.275368,-74.006027,9.4
45188,2018-12-21T00:00:00,TMAX,GHCND:USW00014732,",,W,2400",16.7,"LAGUARDIA AIRPORT, NY US",40.779440,-73.880350,3.4
59823,2018-01-15T00:00:00,WDF5,GHCND:USW00094741,",,W,",40.0,"TETERBORO AIRPORT, NJ US",40.850000,-74.061390,2.7
10852,2018-10-31T00:00:00,PRCP,GHCND:US1NJMN0069,"T,,N,",0.0,"LONG BRANCH 1.7 SSW, NJ US",40.275368,-74.006027,9.4
46755,2018-05-05T00:00:00,SNOW,GHCND:USW00014734,",,W,",0.0,"NEWARK LIBERTY INTERNATIONAL AIRPORT, NJ US",40.682500,-74.169400,2.1


We are losing stations that don't have weather observations associated with them, if we don't want to lose these rows, we perform a right or left join instead of the inner join:

In [ ]:
# how='left' -> 왼쪽 표(station_info)의 행은 전부 남김. 짝이 없으면 오른쪽 값은 NaN
left_join = station_info.merge(weather, left_on='id', right_on='station', how='left')
# how='right' -> 오른쪽 표(station_info)의 행은 전부 남김. 위와 표 순서만 바뀌었고 결과는 같음
right_join = weather.merge(station_info, left_on='station', right_on='id', how='right')

# right_join.datatype.isna() -> 날씨 데이터가 NaN인 행 = 날씨 기록이 없는 관측소들
# inner join에서는 이 관측소들이 아예 사라졌는데, right join에서는 NaN으로 남아있음
right_join[right_join.datatype.isna()].head()


,date,datatype,station,attributes,value,id,name,latitude,longitude,elevation
0,NaN,NaN,NaN,NaN,NaN,GHCND:US1CTFR0022,"STAMFORD 2.6 SSW, CT US",41.064100,-73.577000,36.6
344,NaN,NaN,NaN,NaN,NaN,GHCND:US1NJBG0001,"BERGENFIELD 0.3 SW, NJ US",40.921298,-74.001983,20.1
345,NaN,NaN,NaN,NaN,NaN,GHCND:US1NJBG0002,"SADDLE BROOK TWP 0.6 E, NJ US",40.902694,-74.083358,16.8
718,NaN,NaN,NaN,NaN,NaN,GHCND:US1NJBG0005,"WESTWOOD 0.8 ESE, NJ US",40.983041,-74.015858,15.8
719,NaN,NaN,NaN,NaN,NaN,GHCND:US1NJBG0006,"RAMSEY 0.6 E, NJ US",41.058611,-74.134068,112.2


The left and right join as we performed above are equivalent because the side for which we kept the rows without matches was the same in both cases:

In [ ]:
# left_join과 right_join이 정말 같은지 확인
# .sort_index(axis=1) -> 열 순서를 알파벳순으로 (두 표의 열 순서가 달라서 맞춰줘야 함)
# .sort_values(['date', 'station'], ignore_index=True) -> 행 순서도 맞추고 인덱스 새로 매김
# -> True. 어느 쪽 표를 '전부 남길지'가 같으면 left든 right든 결과는 같음
left_join.sort_index(axis=1).sort_values(['date', 'station'], ignore_index=True).equals(
    right_join.sort_index(axis=1).sort_values(['date', 'station'], ignore_index=True)
)


True

Note we have additional rows in the left and right joins because we kept all the stations that didn't have weather observations:

In [ ]:
# 행 개수 비교: inner는 78780, left/right는 78949
# 차이 169 = 날씨 기록 없는 관측소 169개가 NaN 행으로 추가됨
get_row_count(inner_join, left_join, right_join)


[78780, 78949, 78949]

If we query the station information for stations that have `US1NY` in their ID and perform an outer join, we can see where the mismatches occur:

In [ ]:
# how='outer' -> 양쪽 표의 행을 전부 남김 (짝 없으면 NaN)
# station_info[station_info.id.str.contains('US1NY')] -> 뉴욕(US1NY) 관측소 정보만 오른쪽 표로
# indicator=True -> _merge 라는 열이 추가돼서 각 행이 어디서 왔는지 표시해줌
#   'both' = 양쪽에 다 있음 / 'left_only' = 왼쪽(weather)에만 / 'right_only' = 오른쪽(station_info)에만
outer_join = weather.merge(
    station_info[station_info.id.str.contains('US1NY')],
    left_on='station', right_on='id', how='outer', indicator=True
)

# _merge 값 종류별로 2행씩 뽑아서 보기
# outer_join._merge.unique() -> both, left_only, right_only
# f'_merge == "{kind}"' -> f-string. 문자열 앞에 f 붙이면 {} 안에 변수를 끼워 넣을 수 있음. kind가 'both'면 '_merge == "both"'가 됨
# 각각 query로 골라서 sample(2) -> pd.concat으로 합침 -> sort_index로 정렬
# 결과: both는 뉴욕 관측소의 날씨, left_only는 뉴욕 아닌 관측소 날씨(오른쪽 정보 NaN), right_only는 날씨 기록 없는 뉴욕 관측소(왼쪽 NaN)
pd.concat([
    outer_join.query(f'_merge == "{kind}"').sample(2, random_state=0)
    for kind in outer_join._merge.unique()
]).sort_index()


,date,datatype,station,attributes,value,id,name,latitude,longitude,elevation,_merge
23634,2018-04-12T00:00:00,PRCP,GHCND:US1NYNS0043,",,N,",0.0,GHCND:US1NYNS0043,"PLAINVIEW 0.4 ENE, NY US",40.785919,-73.466873,56.7,both
25742,2018-03-25T00:00:00,PRCP,GHCND:US1NYSF0061,",,N,",0.0,GHCND:US1NYSF0061,"CENTERPORT 0.9 SW, NY US",40.891689,-73.383133,53.6,both
60645,2018-04-16T00:00:00,TMIN,GHCND:USW00094741,",,W,",3.9,NaN,NaN,NaN,NaN,NaN,left_only
70764,2018-03-23T00:00:00,SNWD,GHCND:US1NJHD0002,",,N,",203.0,NaN,NaN,NaN,NaN,NaN,left_only
78790,NaN,NaN,NaN,NaN,NaN,GHCND:US1NYQN0033,"HOWARD BEACH 0.4 NNW, NY US",40.662099,-73.841345,2.1,right_only
78800,NaN,NaN,NaN,NaN,NaN,GHCND:US1NYWC0009,"NEW ROCHELLE 1.3 S, NY US",40.904000,-73.777000,21.9,right_only


These joins are equivalent to their SQL counterparts. Below is the inner join. Note that to use `equals()` you will have to do some manipulation of the dataframes to line them up:

In [ ]:
import sqlite3

# 같은 inner join을 SQL로. JOIN stations ON weather.station == stations.id -> 두 테이블을 station=id 기준으로 합쳐
with sqlite3.connect('data/weather.db') as connection:
    inner_join_from_db = pd.read_sql(
        'SELECT * FROM weather JOIN stations ON weather.station == stations.id',
        connection
    )

# 크기(행, 열)가 pandas merge 결과와 같은지 -> True
inner_join_from_db.shape == inner_join.shape


True

Revisiting the dirty data from chapter 3's [`5-handling_data_issues.ipynb`](../ch_03/5-handling_data_issues.ipynb) notebook.

Data meanings:
- `PRCP`: precipitation in millimeters
- `SNOW`: snowfall in millimeters
- `SNWD`: snow depth in millimeters
- `TMAX`: maximum daily temperature in Celsius
- `TMIN`: minimum daily temperature in Celsius
- `TOBS`: temperature at time of observation in Celsius
- `WESF`: water equivalent of snow in millimeters


Read in the data, dropping duplicates and the uninformative `SNWD` column:

In [ ]:
# 3장에서 봤던 더러운 날씨 데이터 다시 읽기
# index_col='date' -> date를 인덱스로 / .drop_duplicates() -> 완전히 똑같은 중복 행 제거 / .drop(columns='SNWD') -> 쓸모없는 SNWD 열 삭제
dirty_data = pd.read_csv(
    'data/dirty_data.csv', index_col='date'
).drop_duplicates().drop(columns='SNWD')
dirty_data.head()


,station,PRCP,SNOW,TMAX,TMIN,TOBS,WESF,inclement_weather
date,,,,,,,,
2018-01-01T00:00:00,?,0.0,0.0,5505.0,-40.0,NaN,NaN,NaN
2018-01-02T00:00:00,GHCND:USC00280907,0.0,0.0,-8.3,-16.1,-12.2,NaN,False
2018-01-03T00:00:00,GHCND:USC00280907,0.0,0.0,-4.4,-13.9,-13.3,NaN,False
2018-01-04T00:00:00,?,20.6,229.0,5505.0,-40.0,NaN,19.3,True
2018-01-05T00:00:00,?,0.3,NaN,5505.0,-40.0,NaN,NaN,NaN


We need to create two dataframes for the join. We will drop some unecessary columns as well for easier viewing:

In [ ]:
# merge 연습용으로 표를 두 개로 쪼갬
# valid_station -> 진짜 관측소(station != "?") 행만. WESF, station 열은 뺌
valid_station = dirty_data.query('station != "?"').drop(columns=['WESF', 'station'])
# station_with_wesf -> '?' 관측소 행만 (WESF 값은 여기에만 있었음). station, 기온 열들은 뺌
station_with_wesf = dirty_data.query('station == "?"').drop(columns=['station', 'TOBS', 'TMIN', 'TMAX'])


Our column for the join is the index in both dataframes, so we must specify `left_index` and `right_index`:

In [ ]:
# 두 표 모두 date가 '인덱스'라서 열 이름 대신 left_index=True, right_index=True 로 "인덱스끼리 맞춰"라고 알려줌
# how='left' -> valid_station 행은 전부 남김
# .query('WESF > 0') → 합친 결과 중 WESF가 0보다 큰 행만
# 결과 열 이름을 보면 PRCP_x, PRCP_y, SNOW_x, SNOW_y 처럼 됨
# -> 양쪽에 같은 이름의 열(PRCP, SNOW, inclement_weather)이 있어서 pandas가 자동으로 _x(왼쪽), _y(오른쪽)를 붙여 구분한 거임
valid_station.merge(
    station_with_wesf, how='left', left_index=True, right_index=True
).query('WESF > 0').head()


,PRCP_x,SNOW_x,TMAX,TMIN,TOBS,inclement_weather_x,PRCP_y,SNOW_y,WESF,inclement_weather_y
date,,,,,,,,,,
2018-01-30T00:00:00,0.0,0.0,6.7,-1.7,-0.6,False,1.5,13.0,1.8,True
2018-03-08T00:00:00,48.8,NaN,1.1,-0.6,1.1,False,28.4,NaN,28.7,NaN
2018-03-13T00:00:00,4.1,51.0,5.6,-3.9,0.0,True,3.0,13.0,3.0,True
2018-03-21T00:00:00,0.0,0.0,2.8,-2.8,0.6,False,6.6,114.0,8.6,True
2018-04-02T00:00:00,9.1,127.0,12.8,-1.1,-1.1,True,14.0,152.0,15.2,True


The columns that existed in both dataframes, but didn't form part of the join got suffixes added to their names: `_x` for columns from the left dataframe and `_y` for columns from the right dataframe. We can customize this with the `suffixes` argument:

In [ ]:
# suffixes=('', '_?') -> 겹치는 열 이름에 붙일 꼬리표를 직접 정함. 왼쪽은 그대로(''), 오른쪽은 '_?'
# -> PRCP(진짜 관측소), PRCP_?('?' 관측소) 이렇게 알아보기 쉬워짐
valid_station.merge(
    station_with_wesf, how='left', left_index=True, right_index=True, suffixes=('', '_?')
).query('WESF > 0').head()


,PRCP,SNOW,TMAX,TMIN,TOBS,inclement_weather,PRCP_?,SNOW_?,WESF,inclement_weather_?
date,,,,,,,,,,
2018-01-30T00:00:00,0.0,0.0,6.7,-1.7,-0.6,False,1.5,13.0,1.8,True
2018-03-08T00:00:00,48.8,NaN,1.1,-0.6,1.1,False,28.4,NaN,28.7,NaN
2018-03-13T00:00:00,4.1,51.0,5.6,-3.9,0.0,True,3.0,13.0,3.0,True
2018-03-21T00:00:00,0.0,0.0,2.8,-2.8,0.6,False,6.6,114.0,8.6,True
2018-04-02T00:00:00,9.1,127.0,12.8,-1.1,-1.1,True,14.0,152.0,15.2,True


Since we are joining on the index, an easier way is to use the `join()` method instead of `merge()`. Note that the suffix parameter is now `lsuffix` for the left dataframe's suffix and `rsuffix` for the right one's:

In [ ]:
# .join() -> 인덱스 기준으로 합칠 때 쓰는 merge의 간편 버전. left_index/right_index 안 써도 됨
# rsuffix='_?' -> 오른쪽(right) 표의 겹치는 열에 붙일 꼬리표 (왼쪽은 lsuffix)
# 결과는 바로 위 merge와 똑같음. 인덱스로 합칠 땐 join이 더 짧음
valid_station.join(station_with_wesf, how='left', rsuffix='_?').query('WESF > 0').head()


,PRCP,SNOW,TMAX,TMIN,TOBS,inclement_weather,PRCP_?,SNOW_?,WESF,inclement_weather_?
date,,,,,,,,,,
2018-01-30T00:00:00,0.0,0.0,6.7,-1.7,-0.6,False,1.5,13.0,1.8,True
2018-03-08T00:00:00,48.8,NaN,1.1,-0.6,1.1,False,28.4,NaN,28.7,NaN
2018-03-13T00:00:00,4.1,51.0,5.6,-3.9,0.0,True,3.0,13.0,3.0,True
2018-03-21T00:00:00,0.0,0.0,2.8,-2.8,0.6,False,6.6,114.0,8.6,True
2018-04-02T00:00:00,9.1,127.0,12.8,-1.1,-1.1,True,14.0,152.0,15.2,True


Joins can be very resource-intensive, so it's a good idea to figure out what type of join you need using set operations before trying the join itself. The `pandas` set operations are performed on the index, so whichever columns we will be joining on will need to be the index. Let's go back to the `weather` and `station_info` dataframes and set the station ID columns as the index:

In [ ]:
# merge는 데이터가 크면 무거운 작업이라, 하기 전에 "뭐가 겹치고 뭐가 빠질지" 미리 보는 게 좋음
# 그러려면 합칠 기준 열을 인덱스로 만들어야 함 (pandas 집합 연산은 인덱스끼리 함)
weather.set_index('station', inplace=True)
station_info.set_index('id', inplace=True)


The intersection will tell us the stations that are present in both dataframes. The result will be the index when performing an inner join:

In [ ]:
# .intersection() -> 교집합. 양쪽 인덱스에 모두 있는 관측소들
# = inner join 했을 때 남는 관측소들 (110개)
weather.index.intersection(station_info.index)


Index(['GHCND:US1CTFR0039', 'GHCND:US1NJBG0015', 'GHCND:US1NJBG0017',
       'GHCND:US1NJBG0018', 'GHCND:US1NJBG0023', 'GHCND:US1NJBG0030',
       'GHCND:US1NJBG0039', 'GHCND:US1NJBG0044', 'GHCND:US1NJES0018',
       'GHCND:US1NJES0024',
       ...
       'GHCND:US1NJBG0037', 'GHCND:USC00284987', 'GHCND:US1NJES0031',
       'GHCND:US1NJES0029', 'GHCND:US1NJMD0086', 'GHCND:US1NJMS0097',
       'GHCND:US1NJMN0081', 'GHCND:US1NJMD0088', 'GHCND:US1NJES0040',
       'GHCND:US1NYQN0029'],
      dtype='object', length=110)

The set difference will tell us what we lose from each side. When performing an inner join, we lose nothing from the `weather` dataframe:

In [ ]:
# .difference() -> 차집합. weather에는 있는데 station_info에는 없는 관측소
# -> 비어있음(Index([])) = inner join 해도 weather 쪽에서 잃는 건 없음
weather.index.difference(station_info.index)


Index([], dtype='object')

We lose 169 stations from the `station_info` dataframe, however:

In [ ]:
# 반대로 station_info에는 있는데 weather에는 없는 관측소 -> 169개
# = inner join 하면 이 169개 관측소가 사라진다는 뜻 (앞에서 left/right join 행 수 차이 169와 일치!)
station_info.index.difference(weather.index)


Index(['GHCND:US1CTFR0022', 'GHCND:US1NJBG0001', 'GHCND:US1NJBG0002',
       'GHCND:US1NJBG0005', 'GHCND:US1NJBG0006', 'GHCND:US1NJBG0008',
       'GHCND:US1NJBG0011', 'GHCND:US1NJBG0012', 'GHCND:US1NJBG0013',
       'GHCND:US1NJBG0020',
       ...
       'GHCND:USC00308322', 'GHCND:USC00308749', 'GHCND:USC00308946',
       'GHCND:USC00309117', 'GHCND:USC00309270', 'GHCND:USC00309400',
       'GHCND:USC00309466', 'GHCND:USC00309576', 'GHCND:USW00014708',
       'GHCND:USW00014786'],
      dtype='object', length=169)

The symmetric difference tells us what we lose from both sides. It is the combination of the set differences in each direction:

In [ ]:
# 뉴욕 관측소 정보만 따로 뽑음
ny_in_name = station_info[station_info.index.str.contains('US1NY')]

# .symmetric_difference() -> 대칭 차집합. 한쪽에만 있는 것들 (양쪽 방향 차집합을 합친 것)
# (A에만 있는 개수) + (B에만 있는 개수) == (대칭 차집합 개수) 인지 확인 -> True
# \ 는 줄 이어쓰기
ny_in_name.index.difference(weather.index).shape[0]\
+ weather.index.difference(ny_in_name.index).shape[0]\
== weather.index.symmetric_difference(ny_in_name.index).shape[0]


True

The union will show us everything that will be present after a full outer join. Note that we pass in the unique values of the index to make sure we can see the number of stations we will be left with:

In [ ]:
# .union() -> 합집합. 양쪽에 있는 관측소 전부 (중복은 하나로)
# = outer join 했을 때 남는 관측소들
# weather.index.unique() -> weather 인덱스는 같은 관측소가 여러 번 있으니 중복 제거하고 나서 합집합
weather.index.unique().union(station_info.index)


Index(['GHCND:US1CTFR0022', 'GHCND:US1CTFR0039', 'GHCND:US1NJBG0001',
       'GHCND:US1NJBG0002', 'GHCND:US1NJBG0003', 'GHCND:US1NJBG0005',
       'GHCND:US1NJBG0006', 'GHCND:US1NJBG0008', 'GHCND:US1NJBG0010',
       'GHCND:US1NJBG0011',
       ...
       'GHCND:USW00014708', 'GHCND:USW00014732', 'GHCND:USW00014734',
       'GHCND:USW00014786', 'GHCND:USW00054743', 'GHCND:USW00054787',
       'GHCND:USW00094728', 'GHCND:USW00094741', 'GHCND:USW00094745',
       'GHCND:USW00094789'],
      dtype='object', length=279)

Note that the symmetric difference is actually the union of the set differences:

In [ ]:
ny_in_name = station_info[station_info.index.str.contains('US1NY')]

# 대칭 차집합 = (A-B) 합집합 (B-A) 라는 걸 확인 -> True
# 집합 연산 정리: intersection(교집합) = inner / union(합집합) = outer / difference(차집합) = 잃는 것
ny_in_name.index.difference(weather.index).union(weather.index.difference(ny_in_name.index)).equals(
    weather.index.symmetric_difference(ny_in_name.index)
)


True

<hr>
<div>
    <a href="../ch_03/5-handling_data_issues.ipynb">
        <button>&#8592; Chapter 3</button>
    </a>
    <a href="./0-weather_data_collection.ipynb">
        <button>Weather Data Collection</button>
    </a>
    <a href="./2-dataframe_operations.ipynb">
        <button style="float: right;">Next Notebook &#8594;</button>
    </a>
</div>
<hr>